# Intelligent University Observatory


## 1. Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers networkx matplotlib requests

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import requests
import sqlite3
import networkx as nx
import matplotlib.pyplot as plt

from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## 3. Observer Agent (Real Data Collection)

In [ ]:
class AgentObserver:
    def fetch_papers(self, query="machine learning", limit=100):
        url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {
            "query": query,
            "limit": limit,
            "fields": "title,authors,year,citationCount,fieldsOfStudy"
        }
        response = requests.get(url, params=params)
        data = response.json()

        papers = []
        for p in data.get("data", []):
            papers.append({
                "title": p.get("title"),
                "year": p.get("year"),
                "citations": p.get("citationCount"),
                "authors": [a["name"] for a in p.get("authors", [])],
                "fields": p.get("fieldsOfStudy")
            })
        return pd.DataFrame(papers)

## 4. Build Researcher Dataset

In [ ]:
def build_researchers(papers_df):
    from collections import defaultdict

    researchers = defaultdict(lambda: {
        "name": "",
        "publications_count": 0,
        "citations": 0,
        "expertise": []
    })

    for _, row in papers_df.iterrows():
        for author in row["authors"]:
            researchers[author]["name"] = author
            researchers[author]["publications_count"] += 1
            researchers[author]["citations"] += row.get("citations", 0) or 0

            if row.get("fields"):
                researchers[author]["expertise"].extend(row["fields"])

    # 🔥 Build full dataframe properly
    df = pd.DataFrame([
        {
            "name": v["name"],
            "publications_count": v["publications_count"],
            "citations": v["citations"],
            "expertise": ", ".join(set(v["expertise"])) if v["expertise"] else "AI"
        }
        for v in researchers.values()
    ])

    # 🔥 IMPORTANT: ensure all columns exist
    for col in ["name", "publications_count", "citations", "expertise"]:
        if col not in df.columns:
            df[col] = 0

    return df[df["publications_count"] > 1]

## 5. Profiling Agent

In [ ]:
class AgentProfiler:
    def run(self, df):
        print("Top Researchers by Citations:")
        display(df.sort_values(by="citations", ascending=False).head(10))

## 6. Database

In [ ]:
conn = sqlite3.connect("research.db")

def save_database(researchers, papers):
    import sqlite3

    conn = sqlite3.connect("research.db")

    papers_fixed = papers.copy()

    # --- SAFE FIX: handle missing columns ---
    if "authors" in papers_fixed.columns:
        papers_fixed["authors"] = papers_fixed["authors"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else str(x)
        )
    else:
        papers_fixed["authors"] = "Unknown"

    if "fields" in papers_fixed.columns:
        papers_fixed["fields"] = papers_fixed["fields"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else str(x)
        )
    else:
        papers_fixed["fields"] = "Unknown"

    # --- Save tables ---
    researchers.to_sql("Researchers", conn, if_exists="replace", index=False)
    papers_fixed.to_sql("Publications", conn, if_exists="replace", index=False)

    labs = pd.DataFrame({
        "lab_id": range(len(researchers)),
        "lab_name": ["AI Lab"] * len(researchers),
        "researcher_name": researchers["name"]
    })

    labs.to_sql("Labs", conn, if_exists="replace", index=False)

    print("Database saved successfully")

## 7. Expertise Matching

In [ ]:
class AgentExpertiseMatcher:
    def run(self, df):
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode(df["expertise"].tolist())
        sim = cosine_similarity(embeddings)
        return embeddings, sim

## 8. Clustering

In [ ]:
class AgentCluster:
    def run(self, df, embeddings):
        kmeans = KMeans(n_clusters=5, random_state=42)
        df["cluster"] = kmeans.fit_predict(embeddings)
        return df

## 9. Collaboration Recommendation

In [ ]:
def compute_score(r1, r2, sim):
    impact = (r1["citations"] + r2["citations"]) / 20000

    pub1 = r1.get("publications_count", 0)
    pub2 = r2.get("publications_count", 0)

    productivity = (pub1 + pub2) / 100

    complementarity = 1 - sim

    return sim*0.5 + impact*0.3 + productivity*0.2 + complementarity*0.2

class AgentCollabAdvisor:
    def run(self, df, sim):
        results = []
        for i in range(len(df)):
            for j in range(i+1, len(df)):
                r1 = df.iloc[i]
                r2 = df.iloc[j]
                score = compute_score(r1, r2, sim[i][j])
                results.append({
                    "r1": r1["name"],
                    "r2": r2["name"],
                    "score": score
                })
        return pd.DataFrame(results).sort_values(by="score", ascending=False)

## 10. Negotiation (Game Theory)

In [ ]:
class AgentNegotiator:
    def run(self, df):
        decisions = []
        for _, row in df.iterrows():
            benefit = row["score"]
            cost = np.random.uniform(0.2, 0.5)
            payoff = benefit - cost
            decisions.append("ACCEPTED" if payoff > 0.3 else "REJECTED")
        df["decision"] = decisions
        return df

## 11. Visualization

In [ ]:
def plot_network(df):
    G = nx.Graph()
    for _, row in df.head(30).iterrows():
        if row["decision"] == "ACCEPTED":
            G.add_edge(row["r1"], row["r2"], weight=row["score"])

    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_size=2000, font_size=7)
    plt.show()

## 12. Run Full Pipeline

In [ ]:
observer = AgentObserver()
papers = observer.fetch_papers()

researchers = build_researchers(papers)

print("Columns:", researchers.columns)

if "publications_count" not in researchers.columns:
    raise Exception("Column missing — check build_researchers()")

profiler = AgentProfiler()
profiler.run(researchers)

save_database(researchers, papers)

matcher = AgentExpertiseMatcher()
embeddings, sim = matcher.run(researchers)

cluster = AgentCluster()
clustered = cluster.run(researchers, embeddings)

advisor = AgentCollabAdvisor()
collabs = advisor.run(clustered, sim)

print("Top Collaborations:")
display(collabs.head(10))

negotiator = AgentNegotiator()
final = negotiator.run(collabs)

print("Accepted Collaborations:")
display(final[final["decision"] == "ACCEPTED"].head(10))

plot_network(final)

final.to_csv("final_results.csv", index=False)
print("CSV exported successfully")

## Conclusion
This notebook demonstrates a complete Multi-Agent System using real-world data, semantic AI, clustering, collaboration optimization, and game theory.

In [ ]:
final.to_csv("final_results.csv", index=False)